# 2-hour temporal resolution

# Zip files adjusts

In [ ]:
# Download Nagoya_TEC_maps_intersection_raw_files_2022_2024.zip from https://doi.org/10.5281/zenodo.15453941

In [ ]:
%pwd

'/content'

In [ ]:
!ls

Nagoya_TEC_maps_intersection_raw_files_2022_2024.zip  sample_data


In [19]:
!unzip -q 'Nagoya_TEC_maps_intersection_raw_files_2022_2024.zip'

# 2022

In [20]:
!pip install netCDF4

In [21]:
!pip install xarray

In [22]:
import xarray as xr

In [23]:
%pwd

'/content'

In [24]:
!ls

2022030100_atec.nc  2023090113_atec.nc
2022030101_atec.nc  2023090114_atec.nc
2022030102_atec.nc  2023090115_atec.nc
2022030103_atec.nc  2023090116_atec.nc
2022030104_atec.nc  2023090117_atec.nc
2022030105_atec.nc  2023090118_atec.nc
2022030106_atec.nc  2023090119_atec.nc
2022030107_atec.nc  2023090120_atec.nc
2022030108_atec.nc  2023090121_atec.nc
2022030109_atec.nc  2023090122_atec.nc
2022030110_atec.nc  2023090123_atec.nc
2022030111_atec.nc  2023090200_atec.nc
2022030112_atec.nc  2023090201_atec.nc
2022030113_atec.nc  2023090202_atec.nc
2022030114_atec.nc  2023090203_atec.nc
2022030115_atec.nc  2023090204_atec.nc
2022030116_atec.nc  2023090205_atec.nc
2022030117_atec.nc  2023090206_atec.nc
2022030118_atec.nc  2023090207_atec.nc
2022030119_atec.nc  2023090208_atec.nc
2022030120_atec.nc  2023090209_atec.nc
2022030121_atec.nc  2023090210_atec.nc
2022030122_atec.nc  2023090211_atec.nc
2022030123_atec.nc  2023090212_atec.nc
2022030200_atec.nc  2023090213_atec.nc
2022030201_atec.nc  20230

In [25]:
import os
import glob
from collections import defaultdict

base_path = './'

file_pattern = os.path.join(base_path, '**', '2022*.nc')
file_list = sorted(glob.glob(file_pattern, recursive=True))

file_count_by_day = defaultdict(int)

for file_name in file_list:

    day_dir = os.path.basename(os.path.dirname(file_name))
    file_count_by_day[day_dir] += 1

for day, count in sorted(file_count_by_day.items()):
    print(f"Dia {day}: {count} arquivos")

print("\nDays with less than 144 files:")

for day, count in sorted(file_count_by_day.items()):
    if count < 144:
        print(f"Dia {day}: {count} arquivos")

Dia .: 2304 arquivos

Days with less than 144 files:


In [26]:
# !ls

In [27]:
import numpy as np
import xarray as xr
import os
import re
import glob

file_list = glob.glob("./2022*.nc")

pattern = re.compile(r'(\d{8})(\d{2})_atec')

def extract_datetime(file_name):
    match = pattern.search(file_name)
    if match:
        date = match.group(1)
        hour = match.group(2)
        return date, int(hour)
    return None, None

files_by_date = {}

for file_name in file_list:
    date, hour = extract_datetime(file_name)
    if date:
        if date not in files_by_date:
            files_by_date[date] = []
        files_by_date[date].append((file_name, hour))

sorted_dates = sorted(files_by_date.keys())
print(f"\nDates to be processed in chronological order: {sorted_dates}")

print("\nProcessing data by day (in chronological order)...")
for date in sorted_dates:
    file_hour_pairs = files_by_date[date]

    file_hour_pairs.sort(key=lambda x: x[1])

    print(f"Processing day: {date}, {len(file_hour_pairs)} files found")

    final_data = np.zeros((288, 242, 221))

    for file_name, hour in file_hour_pairs:
        try:
            print(f"  Processing: {file_name}, Hora: {hour}")

            ds_disk = xr.open_dataset(file_name)
            tec = ds_disk['atec'][19:261, 140:361].values

            for timestep in range(12):

                global_idx = hour * 12 + timestep

                timestep_slice = tec[:, :, timestep]

                final_data[global_idx] = timestep_slice

            ds_disk.close()

        except Exception as e:
            print(f"  Error processing file {file_name}: {e}")

    print(f"Final shape of the array: {final_data.shape}")

    output_file = f"atec{date}.npy"
    np.save(output_file, final_data)

    print(f"File {output_file} saved successfully! Shape: {final_data.shape}")

print("\Processing completed!")


Dates to be processed in chronological order: ['20220301', '20220302', '20220303', '20220304', '20220305', '20220306', '20220307', '20220308', '20220310', '20220311', '20220312', '20220314', '20220315', '20220316', '20220317', '20220318', '20220319', '20220320', '20220321', '20220322', '20220323', '20220324', '20220325', '20220326', '20220327', '20220331', '20220601', '20220602', '20220603', '20220605', '20220606', '20220607', '20220608', '20220609', '20220610', '20220611', '20220612', '20220613', '20220614', '20220615', '20220616', '20220617', '20220618', '20220619', '20220620', '20220621', '20220622', '20220623', '20220625', '20220627', '20220630', '20220902', '20220903', '20220904', '20220905', '20220906', '20220907', '20220908', '20220909', '20220910', '20220911', '20220912', '20220913', '20220915', '20220917', '20220918', '20220919', '20220921', '20220922', '20220923', '20220924', '20220925', '20220926', '20220927', '20220928', '20220929', '20220930', '20221204', '20221205', '202

In [28]:
!ls -lR ./*.npy | wc -l

96


In [29]:
import numpy as np
import glob
import re
from datetime import datetime

npy_files = glob.glob("atec2022*.npy")

def extract_doy(filename):
    match = re.search(r'atec(\d{8})\.npy', filename)
    if match:
        date_str = match.group(1)
        try:
            date_obj = datetime.strptime(date_str, '%Y%m%d')
            return date_obj.timetuple().tm_yday
        except ValueError:
            return 0
    return 0

npy_files.sort(key=extract_doy)

print(f"Found {len(npy_files)} NPY files to process.")

all_data = []

for i, file in enumerate(npy_files):
    doy = extract_doy(file)
    print(f"Processing {file} (DOY {doy}) - {i+1}/{len(npy_files)}")

    data = np.load(file)

    all_data.append(data)

    print(f"  Shape: {data.shape}")

combined_data = np.concatenate(all_data, axis=0)

print(f"\nCombined data. Final shape: {combined_data.shape}")

output_file = "Nagoya_TEC_maps_intersection_2022.npy"
np.save(output_file, combined_data)

print(f"\nFile {output_file} saved successfully!")
print(f"File size: {combined_data.nbytes / (1024**2):.2f} MB")

Found 96 NPY files to process.
Processing atec20220301.npy (DOY 60) - 1/96
  Shape: (288, 242, 221)
Processing atec20220302.npy (DOY 61) - 2/96
  Shape: (288, 242, 221)
Processing atec20220303.npy (DOY 62) - 3/96
  Shape: (288, 242, 221)
Processing atec20220304.npy (DOY 63) - 4/96
  Shape: (288, 242, 221)
Processing atec20220305.npy (DOY 64) - 5/96
  Shape: (288, 242, 221)
Processing atec20220306.npy (DOY 65) - 6/96
  Shape: (288, 242, 221)
Processing atec20220307.npy (DOY 66) - 7/96
  Shape: (288, 242, 221)
Processing atec20220308.npy (DOY 67) - 8/96
  Shape: (288, 242, 221)
Processing atec20220310.npy (DOY 69) - 9/96
  Shape: (288, 242, 221)
Processing atec20220311.npy (DOY 70) - 10/96
  Shape: (288, 242, 221)
Processing atec20220312.npy (DOY 71) - 11/96
  Shape: (288, 242, 221)
Processing atec20220314.npy (DOY 73) - 12/96
  Shape: (288, 242, 221)
Processing atec20220315.npy (DOY 74) - 13/96
  Shape: (288, 242, 221)
Processing atec20220316.npy (DOY 75) - 14/96
  Shape: (288, 242, 221

In [30]:
!ls -lh *.npy

-rw-r--r-- 1 root root 118M Aug  2 19:19 atec20220301.npy
-rw-r--r-- 1 root root 118M Aug  2 19:19 atec20220302.npy
-rw-r--r-- 1 root root 118M Aug  2 19:19 atec20220303.npy
-rw-r--r-- 1 root root 118M Aug  2 19:19 atec20220304.npy
-rw-r--r-- 1 root root 118M Aug  2 19:19 atec20220305.npy
-rw-r--r-- 1 root root 118M Aug  2 19:20 atec20220306.npy
-rw-r--r-- 1 root root 118M Aug  2 19:20 atec20220307.npy
-rw-r--r-- 1 root root 118M Aug  2 19:20 atec20220308.npy
-rw-r--r-- 1 root root 118M Aug  2 19:20 atec20220310.npy
-rw-r--r-- 1 root root 118M Aug  2 19:20 atec20220311.npy
-rw-r--r-- 1 root root 118M Aug  2 19:20 atec20220312.npy
-rw-r--r-- 1 root root 118M Aug  2 19:20 atec20220314.npy
-rw-r--r-- 1 root root 118M Aug  2 19:20 atec20220315.npy
-rw-r--r-- 1 root root 118M Aug  2 19:20 atec20220316.npy
-rw-r--r-- 1 root root 118M Aug  2 19:20 atec20220317.npy
-rw-r--r-- 1 root root 118M Aug  2 19:20 atec20220318.npy
-rw-r--r-- 1 root root 118M Aug  2 19:20 atec20220319.npy
-rw-r--r-- 1 r

# 2023

In [31]:
!ls

2022030100_atec.nc  2023090314_atec.nc
2022030101_atec.nc  2023090315_atec.nc
2022030102_atec.nc  2023090316_atec.nc
2022030103_atec.nc  2023090317_atec.nc
2022030104_atec.nc  2023090318_atec.nc
2022030105_atec.nc  2023090319_atec.nc
2022030106_atec.nc  2023090320_atec.nc
2022030107_atec.nc  2023090321_atec.nc
2022030108_atec.nc  2023090322_atec.nc
2022030109_atec.nc  2023090323_atec.nc
2022030110_atec.nc  2023090400_atec.nc
2022030111_atec.nc  2023090401_atec.nc
2022030112_atec.nc  2023090402_atec.nc
2022030113_atec.nc  2023090403_atec.nc
2022030114_atec.nc  2023090404_atec.nc
2022030115_atec.nc  2023090405_atec.nc
2022030116_atec.nc  2023090406_atec.nc
2022030117_atec.nc  2023090407_atec.nc
2022030118_atec.nc  2023090408_atec.nc
2022030119_atec.nc  2023090409_atec.nc
2022030120_atec.nc  2023090410_atec.nc
2022030121_atec.nc  2023090411_atec.nc
2022030122_atec.nc  2023090412_atec.nc
2022030123_atec.nc  2023090413_atec.nc
2022030200_atec.nc  2023090414_atec.nc
2022030201_atec.nc  20230

In [32]:
import os
import glob
from collections import defaultdict

base_path = './'

file_pattern = os.path.join(base_path, '**', '2023*.nc')
file_list = sorted(glob.glob(file_pattern, recursive=True))

file_count_by_day = defaultdict(int)

for file_name in file_list:

    day_dir = os.path.basename(os.path.dirname(file_name))
    file_count_by_day[day_dir] += 1

for day, count in sorted(file_count_by_day.items()):
    print(f"Dia {day}: {count} arquivos")

print("\nDays with less than 144 files:")

for day, count in sorted(file_count_by_day.items()):
    if count < 144:
        print(f"Dia {day}: {count} arquivos")

Dia .: 2376 arquivos

Days with less than 144 files:


In [33]:
import numpy as np
import xarray as xr
import os
import re
import glob

file_list = glob.glob("./2023*.nc")

pattern = re.compile(r'(\d{8})(\d{2})_atec')

def extract_datetime(file_name):
    match = pattern.search(file_name)
    if match:
        date = match.group(1)
        hour = match.group(2)
        return date, int(hour)
    return None, None

files_by_date = {}

for file_name in file_list:
    date, hour = extract_datetime(file_name)
    if date:
        if date not in files_by_date:
            files_by_date[date] = []
        files_by_date[date].append((file_name, hour))

sorted_dates = sorted(files_by_date.keys())
print(f"\nDates to be processed in chronological order: {sorted_dates}")

print("\nProcessing data by day (in chronological order)...")
for date in sorted_dates:
    file_hour_pairs = files_by_date[date]

    file_hour_pairs.sort(key=lambda x: x[1])

    print(f"Processing day: {date}, {len(file_hour_pairs)} files found")

    final_data = np.zeros((288, 242, 221))

    for file_name, hour in file_hour_pairs:
        try:
            print(f"  Processing: {file_name}, Hora: {hour}")

            ds_disk = xr.open_dataset(file_name)
            tec = ds_disk['atec'][19:261, 140:361].values

            for timestep in range(12):

                global_idx = hour * 12 + timestep

                timestep_slice = tec[:, :, timestep]

                final_data[global_idx] = timestep_slice

            ds_disk.close()

        except Exception as e:
            print(f"  Error processing file {file_name}: {e}")

    print(f"Final shape of the array: {final_data.shape}")

    output_file = f"atec{date}.npy"
    np.save(output_file, final_data)

    print(f"File {output_file} saved successfully! Shape: {final_data.shape}")

print("\Processing completed!")


Dates to be processed in chronological order: ['20230302', '20230303', '20230304', '20230305', '20230306', '20230307', '20230308', '20230309', '20230310', '20230311', '20230312', '20230313', '20230315', '20230316', '20230317', '20230318', '20230319', '20230320', '20230321', '20230322', '20230323', '20230324', '20230325', '20230326', '20230327', '20230328', '20230329', '20230330', '20230331', '20230606', '20230607', '20230608', '20230609', '20230610', '20230611', '20230612', '20230613', '20230614', '20230615', '20230616', '20230617', '20230618', '20230619', '20230620', '20230621', '20230622', '20230623', '20230624', '20230625', '20230626', '20230627', '20230628', '20230629', '20230630', '20230901', '20230902', '20230903', '20230904', '20230906', '20230907', '20230919', '20230920', '20230921', '20230922', '20230923', '20230924', '20230925', '20230926', '20230927', '20230928', '20230929', '20230930', '20231201', '20231203', '20231204', '20231205', '20231207', '20231208', '20231209', '202

In [34]:
!ls -lR ./*.npy | wc -l

196


In [35]:
import numpy as np
import glob
import re
from datetime import datetime

npy_files = glob.glob("atec2023*.npy")

def extract_doy(filename):
    match = re.search(r'atec(\d{8})\.npy', filename)
    if match:
        date_str = match.group(1)
        try:
            date_obj = datetime.strptime(date_str, '%Y%m%d')
            return date_obj.timetuple().tm_yday
        except ValueError:
            return 0
    return 0

npy_files.sort(key=extract_doy)

print(f"Found {len(npy_files)} NPY files to process.")

all_data = []

for i, file in enumerate(npy_files):
    doy = extract_doy(file)
    print(f"Processing {file} (DOY {doy}) - {i+1}/{len(npy_files)}")

    data = np.load(file)

    all_data.append(data)

    print(f"  Shape: {data.shape}")

combined_data = np.concatenate(all_data, axis=0)

print(f"\nCombined data. Final shape: {combined_data.shape}")

output_file = "Nagoya_TEC_maps_intersection_2023.npy"
np.save(output_file, combined_data)

print(f"\nFile {output_file} saved successfully!")
print(f"File size: {combined_data.nbytes / (1024**2):.2f} MB")

Found 99 NPY files to process.
Processing atec20230302.npy (DOY 61) - 1/99
  Shape: (288, 242, 221)
Processing atec20230303.npy (DOY 62) - 2/99
  Shape: (288, 242, 221)
Processing atec20230304.npy (DOY 63) - 3/99
  Shape: (288, 242, 221)
Processing atec20230305.npy (DOY 64) - 4/99
  Shape: (288, 242, 221)
Processing atec20230306.npy (DOY 65) - 5/99
  Shape: (288, 242, 221)
Processing atec20230307.npy (DOY 66) - 6/99
  Shape: (288, 242, 221)
Processing atec20230308.npy (DOY 67) - 7/99
  Shape: (288, 242, 221)
Processing atec20230309.npy (DOY 68) - 8/99
  Shape: (288, 242, 221)
Processing atec20230310.npy (DOY 69) - 9/99
  Shape: (288, 242, 221)
Processing atec20230311.npy (DOY 70) - 10/99
  Shape: (288, 242, 221)
Processing atec20230312.npy (DOY 71) - 11/99
  Shape: (288, 242, 221)
Processing atec20230313.npy (DOY 72) - 12/99
  Shape: (288, 242, 221)
Processing atec20230315.npy (DOY 74) - 13/99
  Shape: (288, 242, 221)
Processing atec20230316.npy (DOY 75) - 14/99
  Shape: (288, 242, 221

In [36]:
!ls -lh *.npy

-rw-r--r-- 1 root root 118M Aug  2 19:19 atec20220301.npy
-rw-r--r-- 1 root root 118M Aug  2 19:19 atec20220302.npy
-rw-r--r-- 1 root root 118M Aug  2 19:19 atec20220303.npy
-rw-r--r-- 1 root root 118M Aug  2 19:19 atec20220304.npy
-rw-r--r-- 1 root root 118M Aug  2 19:19 atec20220305.npy
-rw-r--r-- 1 root root 118M Aug  2 19:20 atec20220306.npy
-rw-r--r-- 1 root root 118M Aug  2 19:20 atec20220307.npy
-rw-r--r-- 1 root root 118M Aug  2 19:20 atec20220308.npy
-rw-r--r-- 1 root root 118M Aug  2 19:20 atec20220310.npy
-rw-r--r-- 1 root root 118M Aug  2 19:20 atec20220311.npy
-rw-r--r-- 1 root root 118M Aug  2 19:20 atec20220312.npy
-rw-r--r-- 1 root root 118M Aug  2 19:20 atec20220314.npy
-rw-r--r-- 1 root root 118M Aug  2 19:20 atec20220315.npy
-rw-r--r-- 1 root root 118M Aug  2 19:20 atec20220316.npy
-rw-r--r-- 1 root root 118M Aug  2 19:20 atec20220317.npy
-rw-r--r-- 1 root root 118M Aug  2 19:20 atec20220318.npy
-rw-r--r-- 1 root root 118M Aug  2 19:20 atec20220319.npy
-rw-r--r-- 1 r

# 2024

In [37]:
!ls

2022030100_atec.nc  2023090616_atec.nc
2022030101_atec.nc  2023090617_atec.nc
2022030102_atec.nc  2023090618_atec.nc
2022030103_atec.nc  2023090619_atec.nc
2022030104_atec.nc  2023090620_atec.nc
2022030105_atec.nc  2023090621_atec.nc
2022030106_atec.nc  2023090622_atec.nc
2022030107_atec.nc  2023090623_atec.nc
2022030108_atec.nc  2023090700_atec.nc
2022030109_atec.nc  2023090701_atec.nc
2022030110_atec.nc  2023090702_atec.nc
2022030111_atec.nc  2023090703_atec.nc
2022030112_atec.nc  2023090704_atec.nc
2022030113_atec.nc  2023090705_atec.nc
2022030114_atec.nc  2023090706_atec.nc
2022030115_atec.nc  2023090707_atec.nc
2022030116_atec.nc  2023090708_atec.nc
2022030117_atec.nc  2023090709_atec.nc
2022030118_atec.nc  2023090710_atec.nc
2022030119_atec.nc  2023090711_atec.nc
2022030120_atec.nc  2023090712_atec.nc
2022030121_atec.nc  2023090713_atec.nc
2022030122_atec.nc  2023090714_atec.nc
2022030123_atec.nc  2023090715_atec.nc
2022030200_atec.nc  2023090716_atec.nc
2022030201_atec.nc  20230

In [38]:
import os
import glob
from collections import defaultdict

base_path = './'

file_pattern = os.path.join(base_path, '**', '2024*.nc')
file_list = sorted(glob.glob(file_pattern, recursive=True))

file_count_by_day = defaultdict(int)

for file_name in file_list:

    day_dir = os.path.basename(os.path.dirname(file_name))
    file_count_by_day[day_dir] += 1

for day, count in sorted(file_count_by_day.items()):
    print(f"Dia {day}: {count} arquivos")

print("\nDays with less than 144 files:")

for day, count in sorted(file_count_by_day.items()):
    if count < 144:
        print(f"Dia {day}: {count} arquivos")

Dia .: 2544 arquivos

Days with less than 144 files:


In [39]:
import numpy as np
import xarray as xr
import os
import re
import glob

file_list = glob.glob("./2024*.nc")

pattern = re.compile(r'(\d{8})(\d{2})_atec')

def extract_datetime(file_name):
    match = pattern.search(file_name)
    if match:
        date = match.group(1)
        hour = match.group(2)
        return date, int(hour)
    return None, None

files_by_date = {}

for file_name in file_list:
    date, hour = extract_datetime(file_name)
    if date:
        if date not in files_by_date:
            files_by_date[date] = []
        files_by_date[date].append((file_name, hour))

sorted_dates = sorted(files_by_date.keys())
print(f"\nDates to be processed in chronological order: {sorted_dates}")

print("\nProcessing data by day (in chronological order)...")
for date in sorted_dates:
    file_hour_pairs = files_by_date[date]

    file_hour_pairs.sort(key=lambda x: x[1])

    print(f"Processing day: {date}, {len(file_hour_pairs)} files found")

    final_data = np.zeros((288, 242, 221))

    for file_name, hour in file_hour_pairs:
        try:
            print(f"  Processing: {file_name}, Hora: {hour}")

            ds_disk = xr.open_dataset(file_name)
            tec = ds_disk['atec'][19:261, 140:361].values

            for timestep in range(12):

                global_idx = hour * 12 + timestep

                timestep_slice = tec[:, :, timestep]

                final_data[global_idx] = timestep_slice

            ds_disk.close()

        except Exception as e:
            print(f"  Error processing file {file_name}: {e}")

    print(f"Final shape of the array: {final_data.shape}")

    output_file = f"atec{date}.npy"
    np.save(output_file, final_data)

    print(f"File {output_file} saved successfully! Shape: {final_data.shape}")

print("\Processing completed!")


Dates to be processed in chronological order: ['20240301', '20240302', '20240303', '20240304', '20240305', '20240306', '20240307', '20240308', '20240309', '20240310', '20240311', '20240312', '20240313', '20240314', '20240315', '20240316', '20240317', '20240318', '20240319', '20240320', '20240321', '20240326', '20240327', '20240328', '20240329', '20240331', '20240601', '20240602', '20240603', '20240604', '20240605', '20240606', '20240607', '20240608', '20240609', '20240610', '20240611', '20240612', '20240613', '20240614', '20240615', '20240616', '20240617', '20240618', '20240619', '20240621', '20240622', '20240623', '20240624', '20240625', '20240628', '20240629', '20240903', '20240904', '20240905', '20240906', '20240907', '20240908', '20240909', '20240910', '20240911', '20240912', '20240913', '20240914', '20240917', '20240918', '20240919', '20240920', '20240921', '20240922', '20240923', '20240924', '20240925', '20240926', '20240927', '20240928', '20240930', '20241201', '20241202', '202

In [40]:
!ls -lR ./*.npy | wc -l

303


In [41]:
import numpy as np
import glob
import re
from datetime import datetime

npy_files = glob.glob("atec2024*.npy")

def extract_doy(filename):
    match = re.search(r'atec(\d{8})\.npy', filename)
    if match:
        date_str = match.group(1)
        try:
            date_obj = datetime.strptime(date_str, '%Y%m%d')
            return date_obj.timetuple().tm_yday
        except ValueError:
            return 0
    return 0

npy_files.sort(key=extract_doy)

print(f"Found {len(npy_files)} NPY files to process.")

all_data = []

for i, file in enumerate(npy_files):
    doy = extract_doy(file)
    print(f"Processing {file} (DOY {doy}) - {i+1}/{len(npy_files)}")

    data = np.load(file)

    all_data.append(data)

    print(f"  Shape: {data.shape}")

combined_data = np.concatenate(all_data, axis=0)

print(f"\nCombined data. Final shape: {combined_data.shape}")

output_file = "Nagoya_TEC_maps_intersection_2024.npy"
np.save(output_file, combined_data)

print(f"\nFile {output_file} saved successfully!")
print(f"File size: {combined_data.nbytes / (1024**2):.2f} MB")

Found 106 NPY files to process.
Processing atec20240301.npy (DOY 61) - 1/106
  Shape: (288, 242, 221)
Processing atec20240302.npy (DOY 62) - 2/106
  Shape: (288, 242, 221)
Processing atec20240303.npy (DOY 63) - 3/106
  Shape: (288, 242, 221)
Processing atec20240304.npy (DOY 64) - 4/106
  Shape: (288, 242, 221)
Processing atec20240305.npy (DOY 65) - 5/106
  Shape: (288, 242, 221)
Processing atec20240306.npy (DOY 66) - 6/106
  Shape: (288, 242, 221)
Processing atec20240307.npy (DOY 67) - 7/106
  Shape: (288, 242, 221)
Processing atec20240308.npy (DOY 68) - 8/106
  Shape: (288, 242, 221)
Processing atec20240309.npy (DOY 69) - 9/106
  Shape: (288, 242, 221)
Processing atec20240310.npy (DOY 70) - 10/106
  Shape: (288, 242, 221)
Processing atec20240311.npy (DOY 71) - 11/106
  Shape: (288, 242, 221)
Processing atec20240312.npy (DOY 72) - 12/106
  Shape: (288, 242, 221)
Processing atec20240313.npy (DOY 73) - 13/106
  Shape: (288, 242, 221)
Processing atec20240314.npy (DOY 74) - 14/106
  Shape:

In [42]:
!ls -lh *.npy

-rw-r--r-- 1 root root 118M Aug  2 19:19 atec20220301.npy
-rw-r--r-- 1 root root 118M Aug  2 19:19 atec20220302.npy
-rw-r--r-- 1 root root 118M Aug  2 19:19 atec20220303.npy
-rw-r--r-- 1 root root 118M Aug  2 19:19 atec20220304.npy
-rw-r--r-- 1 root root 118M Aug  2 19:19 atec20220305.npy
-rw-r--r-- 1 root root 118M Aug  2 19:20 atec20220306.npy
-rw-r--r-- 1 root root 118M Aug  2 19:20 atec20220307.npy
-rw-r--r-- 1 root root 118M Aug  2 19:20 atec20220308.npy
-rw-r--r-- 1 root root 118M Aug  2 19:20 atec20220310.npy
-rw-r--r-- 1 root root 118M Aug  2 19:20 atec20220311.npy
-rw-r--r-- 1 root root 118M Aug  2 19:20 atec20220312.npy
-rw-r--r-- 1 root root 118M Aug  2 19:20 atec20220314.npy
-rw-r--r-- 1 root root 118M Aug  2 19:20 atec20220315.npy
-rw-r--r-- 1 root root 118M Aug  2 19:20 atec20220316.npy
-rw-r--r-- 1 root root 118M Aug  2 19:20 atec20220317.npy
-rw-r--r-- 1 root root 118M Aug  2 19:20 atec20220318.npy
-rw-r--r-- 1 root root 118M Aug  2 19:20 atec20220319.npy
-rw-r--r-- 1 r

In [43]:
from google.colab import files

In [44]:
%pwd

'/content'

In [45]:
from google.colab import files

In [46]:
!ls -lh NAGOYA*.npy

ls: cannot access 'NAGOYA*.npy': No such file or directory


In [47]:
# from google.colab import drive
# drive.mount('/content/drive')

In [48]:
# %cp NAGOYA-2022.npy /content/drive/MyDrive/Nagoya_TEC_maps_intersection_2022.npy

In [49]:
# %cp NAGOYA-2023.npy /content/drive/MyDrive/Nagoya_TEC_maps_intersection_2023.npy

In [50]:
# %cp NAGOYA-2024.npy /content/drive/MyDrive/Nagoya_TEC_maps_intersection_2024.npy

# 5D array - merge 2022-2024 data

In [51]:
import numpy as np

## 2022-2024

In [52]:
%cd './'

/content


In [53]:
import numpy as np

In [54]:
# !ls *.npy

In [55]:
%pwd

'/content'

In [56]:
!ls

2022030100_atec.nc  2023091921_atec.nc
2022030101_atec.nc  2023091922_atec.nc
2022030102_atec.nc  2023091923_atec.nc
2022030103_atec.nc  2023092000_atec.nc
2022030104_atec.nc  2023092001_atec.nc
2022030105_atec.nc  2023092002_atec.nc
2022030106_atec.nc  2023092003_atec.nc
2022030107_atec.nc  2023092004_atec.nc
2022030108_atec.nc  2023092005_atec.nc
2022030109_atec.nc  2023092006_atec.nc
2022030110_atec.nc  2023092007_atec.nc
2022030111_atec.nc  2023092008_atec.nc
2022030112_atec.nc  2023092009_atec.nc
2022030113_atec.nc  2023092010_atec.nc
2022030114_atec.nc  2023092011_atec.nc
2022030115_atec.nc  2023092012_atec.nc
2022030116_atec.nc  2023092013_atec.nc
2022030117_atec.nc  2023092014_atec.nc
2022030118_atec.nc  2023092015_atec.nc
2022030119_atec.nc  2023092016_atec.nc
2022030120_atec.nc  2023092017_atec.nc
2022030121_atec.nc  2023092018_atec.nc
2022030122_atec.nc  2023092019_atec.nc
2022030123_atec.nc  2023092020_atec.nc
2022030200_atec.nc  2023092021_atec.nc
2022030201_atec.nc  20230

In [57]:
%pwd

'/content'

In [58]:
!ls *.npy

atec20220301.npy  atec20230308.npy  atec20240310.npy
atec20220302.npy  atec20230309.npy  atec20240311.npy
atec20220303.npy  atec20230310.npy  atec20240312.npy
atec20220304.npy  atec20230311.npy  atec20240313.npy
atec20220305.npy  atec20230312.npy  atec20240314.npy
atec20220306.npy  atec20230313.npy  atec20240315.npy
atec20220307.npy  atec20230315.npy  atec20240316.npy
atec20220308.npy  atec20230316.npy  atec20240317.npy
atec20220310.npy  atec20230317.npy  atec20240318.npy
atec20220311.npy  atec20230318.npy  atec20240319.npy
atec20220312.npy  atec20230319.npy  atec20240320.npy
atec20220314.npy  atec20230320.npy  atec20240321.npy
atec20220315.npy  atec20230321.npy  atec20240326.npy
atec20220316.npy  atec20230322.npy  atec20240327.npy
atec20220317.npy  atec20230323.npy  atec20240328.npy
atec20220318.npy  atec20230324.npy  atec20240329.npy
atec20220319.npy  atec20230325.npy  atec20240331.npy
atec20220320.npy  atec20230326.npy  atec20240601.npy
atec20220321.npy  atec20230327.npy  atec202406

In [59]:
nagoya_2022 = np.load('Nagoya_TEC_maps_intersection_2022.npy')
print(np.shape(nagoya_2022))
print(type(nagoya_2022))
print(type(nagoya_2022[0]))
print(type(nagoya_2022[0][0]))
print(type(nagoya_2022[0][0][0]))

(27648, 242, 221)
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.float64'>


In [60]:
for i in range(len(nagoya_2022)):
    print(np.shape(nagoya_2022[i]))

(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)

In [61]:
np.shape(nagoya_2022)

(27648, 242, 221)

In [62]:
has_nan = np.any(np.isnan(nagoya_2022))
print(has_nan)

count_nan = np.sum(np.isnan(nagoya_2022))
print(count_nan)

True
955519857


In [63]:
nagoya_2022.shape

(27648, 242, 221)

In [64]:
np.set_printoptions(threshold=np.inf)

In [65]:
print(nagoya_2022[0][0])

[15.24487209 15.35594749 15.62724018 15.62724018 15.62724018 15.62724018
 15.62724018 15.67634392 14.8411026  14.8411026  14.8411026  14.8411026
 14.8411026  14.98174191 14.98174191 14.98174191 14.59172058 14.59172058
 14.67130184 14.91833305 13.34468651 13.34468651 13.34468651 13.6588335
 13.6588335  12.99385834 13.52103233 13.52103233 13.52103233 13.52103233
 13.11270905 13.3315134  13.3315134  13.73480034 13.73480034 13.73480034
 13.73480034 13.73480034 13.73480034 13.73480034 13.73480034 13.73480034
 13.37285614 14.27384949 13.73251057 13.73251057 13.43309975 13.57339764
 13.57339764 12.4174366  12.4174366  11.81858444 11.81858444 11.81858444
 12.12085342 12.12085342 11.11243725 11.11243725 11.11243725 11.11243725
 11.11243725 11.11243725 11.11243725 11.11243725 11.11243725 11.260355
 11.260355   11.31899071 11.31899071 12.35031319 11.20330524 11.20330524
 11.42294884 11.42294884 12.28915691 12.28915691 12.28915691 12.28915691
 12.28915691 12.28915691 12.28915691 12.28915691 12.289

In [66]:
print(nagoya_2022[0][-1])

[17.71973038 17.42378616 17.5975666  17.69400597 17.47002983 17.71816635
 18.09120941 18.16869164 18.38394928 18.42401314 18.21523094 18.11359978
 18.10839844 18.00214958 17.75897789 17.85139656 17.66806984 17.43794441
 17.32384872 17.42040062 17.41195297 17.86724854 17.51272392 17.51272392
 17.20814133 15.96339035 15.96339035 15.99704838 16.45387268 16.44707489
 16.47504234 16.47097397 16.41033745 16.25471497 16.16614723 16.24341393
 16.2390995  16.34074974 17.29003143 17.8953228  18.05400276 17.95985222
 18.51299858 18.50794601 18.61274338 18.79347229 19.25214005 19.36553001
 19.43928719 19.43569946 19.54854012 19.4471283  19.30557442 19.37603188
 19.34049988 19.34606171 19.47274017 19.77859497 19.4078598  19.57077789
 19.1026001  18.69226646 18.7926445  19.06453133 18.96324348 19.09129524
 19.16623116 18.84040642 18.29941177 18.08562088 17.95563316 17.79767609
 17.7942028  17.86902237 17.7371254  17.54969788 17.5285244  17.32747841
 17.25768471 17.1607666  17.01792908 16.8663063  16

In [67]:
# del nagoya_2022

In [68]:
nagoya_2023 = np.load('Nagoya_TEC_maps_intersection_2023.npy')
print(np.shape(nagoya_2023))
print(type(nagoya_2023))
print(type(nagoya_2023[0]))
print(type(nagoya_2023[0][0]))
print(type(nagoya_2023[0][0][0]))

(28512, 242, 221)
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.float64'>


In [69]:
for i in range(len(nagoya_2023)):
    print(np.shape(nagoya_2023[i]))

(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)

In [70]:
np.shape(nagoya_2023)

(28512, 242, 221)

In [71]:
has_nan = np.any(np.isnan(nagoya_2023))
print(has_nan)

count_nan = np.sum(np.isnan(nagoya_2023))
print(count_nan)

True
1033801021


In [72]:
nagoya_2023.shape

(28512, 242, 221)

In [73]:
np.set_printoptions(threshold=np.inf)

In [74]:
print(nagoya_2023[0][0])

[15.15973759 14.71166325 14.80329514 14.53179741 14.53179741 14.59951019
 14.95788574 14.95788574 15.23364639 15.23364639 15.23364639 15.59814262
 15.59322739 15.59322739 15.19001389 15.0228281  15.0228281  15.0228281
 15.81139565 15.58796692 15.58796692 15.45769596 15.45769596 15.45769596
 16.15104675 16.22168159 16.22168159 16.22168159 16.31487274 15.97027397
 15.97027397 15.80700207 15.80700207 15.80700207 15.42304993 15.40575981
 15.40575981 16.03258324 15.21506023 15.21506023 15.21506023 14.89948845
 16.01103783 15.25805378 15.25805378 15.25805378 15.25805378 14.27306747
 15.55848217 15.55848217 15.55848217 15.31729317 15.81931686 15.81931686
 15.81931686 15.81931686 15.81931686 15.81931686 15.81931686 14.94812775
 14.91574097 16.67090034 16.67090034 16.87447739 16.87447739 16.23497963
 16.43117714 16.43117714 15.78882122 15.78882122 16.3240242  15.44096565
 15.44096565 16.20858002 15.32277775 15.32277775 15.32277775 15.32277775
 15.32277775 15.32277775 15.32277775 15.32277775 16.

In [75]:
print(nagoya_2023[0][-1])

[38.70494461 38.73809052 38.82811356 38.83785629 38.76746368 38.72145081
 38.68032837 38.74907684 38.72641754 38.85918045 39.05173111 39.0200386
 39.17493439 39.02206421 39.25550842 39.24547958 39.3382225  39.24137878
 39.69591522 39.483284   38.62474442 37.29073334 36.70582581 36.70582581
 35.65550232 34.81812668 34.7190361  34.88911819 34.88911819 34.86254883
 35.15927505 35.81344986 35.43680573 35.43680573 35.53449249 34.90238953
 34.58323669 33.13740921 32.70286179 32.26264191 32.19209671 32.2133255
 32.20660019 32.13477325 32.23672104 32.36314392 32.34621048 32.28551102
 32.40792084 32.30299377 32.16985703 32.06380463 31.95504951 31.62862778
 31.21489906 30.89899826 30.51278496 30.03732872 29.79911041 29.5525856
 29.32038879 29.10385704 28.74104881 28.63807297 28.34884644 28.15119362
 28.0230484  27.77096367 27.8072319  27.7046814  27.61056519 27.55530739
 27.49187851 27.16868591 27.10092545 26.90221596 26.53254318 26.16445351
 26.05755043 25.78522873 25.5384407  25.48319817 25.44

In [76]:
# del nagoya_2023

In [77]:
nagoya_2024 = np.load('Nagoya_TEC_maps_intersection_2024.npy')
print(np.shape(nagoya_2024))
print(type(nagoya_2024))
print(type(nagoya_2024[0]))
print(type(nagoya_2024[0][0]))
print(type(nagoya_2024[0][0][0]))

(30528, 242, 221)
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.float64'>


In [78]:
for i in range(len(nagoya_2024)):
    print(np.shape(nagoya_2024[i]))

(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)

In [79]:
np.shape(nagoya_2024)

(30528, 242, 221)

In [80]:
has_nan = np.any(np.isnan(nagoya_2024))
print(has_nan)

count_nan = np.sum(np.isnan(nagoya_2024))
print(count_nan)

True
1137490513


In [81]:
nagoya_2024.shape

(30528, 242, 221)

In [82]:
np.set_printoptions(threshold=np.inf)

In [83]:
print(nagoya_2024[0][0])

[25.81098938 25.53123283 25.53123283 25.10790443 25.10790443 25.12040329
 24.5556488  24.5556488  24.5556488  23.49295998 24.44730377 25.39146423
 25.39146423 25.58919716 24.60969543 24.29973602 24.29973602 23.48611069
 23.48611069 24.8562851  24.8562851  25.569561   25.569561   24.14855003
 24.14855003 24.14855003 24.14855003 24.14855003 24.14855003 24.14855003
 24.14855003 23.93475151 23.93475151 23.054039   21.54585648 21.54585648
 22.31619072 22.31619072 22.90351868 22.90351868 23.57149506 23.57149506
 23.57149506 22.20942307 20.65611076 20.65611076 21.86031532 21.86031532
 21.86031532 21.86031532 21.86031532 22.22672462 22.22672462 22.22672462
 22.0849781  22.0849781  22.0849781  22.0849781  22.0849781  21.27880669
 22.5953598  22.5953598  22.36440277 23.78794861 23.78794861 23.78794861
 25.19330978 25.19330978 25.19330978 24.76906395 24.76906395 24.76906395
 24.76906395 24.76906395 25.05768394 25.05768394 25.13592529 25.13592529
 25.13592529 25.13592529 25.80965805 25.93213654 26

In [84]:
print(nagoya_2024[0][-1])

[35.07431793 35.19741821 35.12939453 34.96323395 34.89413071 34.13943863
 33.82780457 33.75049973 33.84936905 33.98156738 33.87095261 33.7453804
 33.87460327 33.75517654 34.04606247 34.26619339 34.45626831 34.54187393
 34.67480087 34.77733231 34.89550781 34.99032593 35.15259933 35.24320602
 35.29004669 35.30674744 35.34588242 35.21751785 35.24026489 34.95431519
 34.75771713 34.55168152 34.30333328 34.01041031 33.68069839 33.34481049
 33.15045547 32.78575516 32.26305008 31.92453384 31.80435753 31.43741608
 31.18464088 31.2144165  31.30561256 31.15472412 31.00075531 31.09196472
 31.05386162 30.98993111 30.6632061  30.54385757 30.33006668 29.887043
 29.49579048 29.49891663 29.30683327 29.22710991 29.13576317 29.27242088
 29.2454586  29.19401741 28.93786621 28.79583168 28.49080658 28.26527786
 27.98602104 27.86810303 27.85188484 27.65360641 27.47672272 27.56991005
 27.6119709  27.70610619 27.80072784 27.49389648 27.19206238 26.99819565
 27.11926842 27.51472473 27.81078339 28.66765213 29.11

In [85]:
nagoya_2022_2024 = np.concatenate([nagoya_2022, nagoya_2023, nagoya_2024], axis=0)
print(nagoya_2022_2024.shape)

(86688, 242, 221)


In [86]:
print(type(nagoya_2022_2024))
print(type(nagoya_2022_2024[0]))
print(type(nagoya_2022_2024[0][0]))
print(type(nagoya_2022_2024[0][0][0]))

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.float64'>


In [87]:
for i in range(len(nagoya_2022_2024)):
    print(np.shape(nagoya_2022_2024[i]))

A saída de streaming foi truncada nas últimas 5000 linhas.
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 221)
(242, 

In [88]:
np.shape(nagoya_2022_2024)

(86688, 242, 221)

In [89]:
has_nan = np.any(np.isnan(nagoya_2022_2024))
print(has_nan)

count_nan = np.sum(np.isnan(nagoya_2022_2024))
print(count_nan)

True
3126811391


In [90]:
np.set_printoptions(threshold=np.inf)

In [91]:
print(nagoya_2022_2024[0][0])

[15.24487209 15.35594749 15.62724018 15.62724018 15.62724018 15.62724018
 15.62724018 15.67634392 14.8411026  14.8411026  14.8411026  14.8411026
 14.8411026  14.98174191 14.98174191 14.98174191 14.59172058 14.59172058
 14.67130184 14.91833305 13.34468651 13.34468651 13.34468651 13.6588335
 13.6588335  12.99385834 13.52103233 13.52103233 13.52103233 13.52103233
 13.11270905 13.3315134  13.3315134  13.73480034 13.73480034 13.73480034
 13.73480034 13.73480034 13.73480034 13.73480034 13.73480034 13.73480034
 13.37285614 14.27384949 13.73251057 13.73251057 13.43309975 13.57339764
 13.57339764 12.4174366  12.4174366  11.81858444 11.81858444 11.81858444
 12.12085342 12.12085342 11.11243725 11.11243725 11.11243725 11.11243725
 11.11243725 11.11243725 11.11243725 11.11243725 11.11243725 11.260355
 11.260355   11.31899071 11.31899071 12.35031319 11.20330524 11.20330524
 11.42294884 11.42294884 12.28915691 12.28915691 12.28915691 12.28915691
 12.28915691 12.28915691 12.28915691 12.28915691 12.289

In [92]:
print(nagoya_2022_2024[0][-1])

[17.71973038 17.42378616 17.5975666  17.69400597 17.47002983 17.71816635
 18.09120941 18.16869164 18.38394928 18.42401314 18.21523094 18.11359978
 18.10839844 18.00214958 17.75897789 17.85139656 17.66806984 17.43794441
 17.32384872 17.42040062 17.41195297 17.86724854 17.51272392 17.51272392
 17.20814133 15.96339035 15.96339035 15.99704838 16.45387268 16.44707489
 16.47504234 16.47097397 16.41033745 16.25471497 16.16614723 16.24341393
 16.2390995  16.34074974 17.29003143 17.8953228  18.05400276 17.95985222
 18.51299858 18.50794601 18.61274338 18.79347229 19.25214005 19.36553001
 19.43928719 19.43569946 19.54854012 19.4471283  19.30557442 19.37603188
 19.34049988 19.34606171 19.47274017 19.77859497 19.4078598  19.57077789
 19.1026001  18.69226646 18.7926445  19.06453133 18.96324348 19.09129524
 19.16623116 18.84040642 18.29941177 18.08562088 17.95563316 17.79767609
 17.7942028  17.86902237 17.7371254  17.54969788 17.5285244  17.32747841
 17.25768471 17.1607666  17.01792908 16.8663063  16

# Dates adjusts

In [93]:
import numpy as np
from datetime import datetime, timedelta

def doy_to_date(year, doy):
    return datetime(year, 1, 1) + timedelta(days=doy - 1)

doys = {
    2022: {
        3: [60, 61, 62, 63, 64, 65, 66, 67, 69, 70,
            71, 73, 74, 75, 76, 77, 78, 79, 80, 81,
            82, 83, 84, 85, 86, 90],
        6: [152, 153, 154, 156, 157, 158, 159, 160,
            161, 162, 163, 164, 165, 166, 167, 168,
            169, 170, 171, 172, 173, 174, 176, 178,
            181],
        9: [245, 246, 247, 248, 249, 250, 251, 252,
            253, 254, 255, 256, 258, 260, 261, 262,
            264, 265, 266, 267, 268, 269, 270, 271,
            272, 273],
        12: [338, 339, 340, 348, 349, 350, 351, 352,
             353, 354, 356, 357, 358, 359, 360, 362,
             363, 364, 365]
    },
    2023: {
        3: [61, 62, 63, 64, 65, 66, 67, 68, 69, 70,
            71, 72, 74, 75, 76, 77, 78, 79, 80, 81,
            82, 83, 84, 85, 86, 87, 88, 89, 90],
        6: [157, 158, 159, 160, 161, 162, 163, 164, 165, 166,
            167, 168, 169, 170, 171, 172, 173, 174, 175, 176,
            177, 178, 179, 180, 181],
        9: [244, 245, 246, 247, 249, 250, 262, 263, 264, 265,
            266, 267, 268, 269, 270, 271, 272, 273],
        12: [335, 337, 338, 339, 341, 342, 343, 344, 345,
             346, 347, 348, 349, 350, 351, 352, 353, 355, 356,
             357, 358, 359, 360, 361, 362, 363, 365]
    },
    2024: {
        3: [61, 62, 63, 64, 65, 66, 67, 68, 69, 70,
            71, 72, 73, 74, 75, 76, 77, 78, 79, 80,
            81, 86, 87, 88, 89, 91],
        6: [153, 154, 155, 156, 157, 158, 159, 160, 161, 162,
            163, 164, 165, 166, 167, 168, 169, 170, 171, 173,
            174, 175, 176, 177, 180, 181],
        9: [247, 248, 249, 250, 251, 252, 253, 254, 255, 256,
            257, 258, 261, 262, 263, 264, 265, 266, 267, 268,
            269, 270, 271, 272, 274],
        12: [336, 337, 338, 339, 340, 341, 342, 343, 344, 345,
             346, 347, 349, 350, 351, 352, 353, 354, 355,
             356, 357, 358, 359, 360, 361, 362, 363, 364, 365]
    }
}

datetime_list = []

for year in doys.keys():
    for month in doys[year].keys():
        for doy in doys[year][month]:
            base_date = doy_to_date(year, doy)
            for hour in range(24):
                for minute in range(0, 60, 5):
                    dt = datetime(base_date.year, base_date.month, base_date.day, hour, minute, 0)
                    datetime_list.append(dt)

nagoya_datetimes_2022_2024 = np.array(datetime_list, dtype='datetime64[s]')

print(f"Total number of datetime points: {len(nagoya_datetimes_2022_2024)}")
print(f"First datetime: {nagoya_datetimes_2022_2024[0]}")
print(f"Last datetime: {nagoya_datetimes_2022_2024[-1]}")

print("\nSample of first 10 datetime entries:")
for dt in nagoya_datetimes_2022_2024[:10]:
    print(dt)

print("\nSample of last 10 datetime entries:")
for dt in nagoya_datetimes_2022_2024[-10:]:
    print(dt)

Total number of datetime points: 86688
First datetime: 2022-03-01T00:00:00
Last datetime: 2024-12-30T23:55:00

Sample of first 10 datetime entries:
2022-03-01T00:00:00
2022-03-01T00:05:00
2022-03-01T00:10:00
2022-03-01T00:15:00
2022-03-01T00:20:00
2022-03-01T00:25:00
2022-03-01T00:30:00
2022-03-01T00:35:00
2022-03-01T00:40:00
2022-03-01T00:45:00

Sample of last 10 datetime entries:
2024-12-30T23:10:00
2024-12-30T23:15:00
2024-12-30T23:20:00
2024-12-30T23:25:00
2024-12-30T23:30:00
2024-12-30T23:35:00
2024-12-30T23:40:00
2024-12-30T23:45:00
2024-12-30T23:50:00
2024-12-30T23:55:00


In [94]:
nagoya_2022_2024.shape

(86688, 242, 221)

In [95]:
print(len(nagoya_2022_2024.tolist()))
print(len(nagoya_2022_2024[0].tolist()))
print(len(nagoya_2022_2024[0][0].tolist()))

86688
242
221


In [96]:
nagoya_2022.shape

(27648, 242, 221)

In [97]:
nagoya_2023.shape

(28512, 242, 221)

In [98]:
nagoya_2024.shape

(30528, 242, 221)

In [99]:
nagoya_datetimes_2022 = np.array([d for d in nagoya_datetimes_2022_2024 if d.astype(object).year == 2022])

In [100]:
len(nagoya_datetimes_2022)

27648

In [101]:
nagoya_datetimes_2023 = np.array([d for d in nagoya_datetimes_2022_2024 if d.astype(object).year == 2023])

In [102]:
len(nagoya_datetimes_2023)

28512

In [103]:
nagoya_datetimes_2024 = np.array([d for d in nagoya_datetimes_2022_2024 if d.astype(object).year == 2024])

In [104]:
len(nagoya_datetimes_2024)

30528

In [105]:
nagoya_datetimes_2022_2024[0:12]

array(['2022-03-01T00:00:00', '2022-03-01T00:05:00',
       '2022-03-01T00:10:00', '2022-03-01T00:15:00',
       '2022-03-01T00:20:00', '2022-03-01T00:25:00',
       '2022-03-01T00:30:00', '2022-03-01T00:35:00',
       '2022-03-01T00:40:00', '2022-03-01T00:45:00',
       '2022-03-01T00:50:00', '2022-03-01T00:55:00'],
      dtype='datetime64[s]')

In [106]:
len(nagoya_datetimes_2022_2024[0:12])

12

In [107]:
len(nagoya_2022_2024[0:12])

12

In [108]:
nagoya_2022_2024_shaped = np.reshape(nagoya_2022_2024, (-1, 242, 221))

In [109]:
np.shape(nagoya_2022_2024_shaped)

(86688, 242, 221)

In [110]:
# np.shape(nagoya_2022_2024_shaped.tolist())

## Create dataframe

In [111]:
nagoya_datetimes_2022_2024.shape

(86688,)

In [112]:
import pandas as pd

data = {
    'DATETIME': nagoya_datetimes_2022_2024,
    'TECMAP': nagoya_2022_2024_shaped.tolist()
}

df_nagoya_maps_2022_2024 = pd.DataFrame(data)

df_nagoya_maps_2022_2024['DATETIME'] = pd.to_datetime(df_nagoya_maps_2022_2024['DATETIME'])

df_nagoya_maps_2022_2024.set_index('DATETIME', inplace=True)

# df_nagoya_maps_2022_2024

In [113]:
df_nagoya_maps_2022_2024 = df_nagoya_maps_2022_2024[(df_nagoya_maps_2022_2024.index.minute == 0) & (df_nagoya_maps_2022_2024.index.hour % 2 == 0)]

df1 = df_nagoya_maps_2022_2024[
    ((df_nagoya_maps_2022_2024.index.year == 2022) & (df_nagoya_maps_2022_2024.index.month.isin([3, 6, 9]))) |
    ((df_nagoya_maps_2022_2024.index.year == 2024) & (df_nagoya_maps_2022_2024.index.month.isin([3, 6, 9, 12])))
]

df3 = pd.DataFrame(data)

df3['DATETIME'] = pd.to_datetime(df3['DATETIME'])

df3.set_index('DATETIME', inplace=True)

df3 = df3[
    ((df3.index.year == 2024) & (df3.index.month.isin([3, 6, 9, 12])))
]

df3 = df3[df3.index.minute % 30 == 0]

print("DataFrame 1:")
print(df1)

print("DataFrame 3:")
print(df3)

DataFrame 1:
                                                                TECMAP
DATETIME                                                              
2022-03-01 00:00:00  [[15.244872093200684, 15.355947494506836, 15.6...
2022-03-01 02:00:00  [[11.9190673828125, 11.787118911743164, 11.787...
2022-03-01 04:00:00  [[11.735207557678223, 11.735207557678223, 11.7...
2022-03-01 06:00:00  [[10.82171630859375, 10.82171630859375, 10.821...
2022-03-01 08:00:00  [[8.598779678344727, 8.598779678344727, 8.5987...
...                                                                ...
2024-12-30 14:00:00  [[15.771679878234863, 16.21746826171875, 15.52...
2024-12-30 16:00:00  [[15.003790855407715, 15.003790855407715, 15.0...
2024-12-30 18:00:00  [[14.162177085876465, 13.393721580505371, 13.0...
2024-12-30 20:00:00  [[18.865320205688477, 18.865320205688477, 14.8...
2024-12-30 22:00:00  [[23.508947372436523, 23.508947372436523, 23.5...

[2196 rows x 1 columns]
DataFrame 3:
                          

In [114]:
df_nagoya_maps_2024 = pd.DataFrame(data)

df_nagoya_maps_2024['DATETIME'] = pd.to_datetime(df_nagoya_maps_2024['DATETIME'])

df_nagoya_maps_2024.set_index('DATETIME', inplace=True)

df_nagoya_maps_2024 = df_nagoya_maps_2024[
    ((df_nagoya_maps_2024.index.year == 2024) & (df_nagoya_maps_2024.index.month.isin([3, 6, 9, 12])))
]

In [148]:
df_nagoya_maps_2024

,TECMAP
DATETIME,
2024-03-01 00:00:00,"[[25.810989379882812, 25.531232833862305, 25.5..."
2024-03-01 00:05:00,"[[25.312646865844727, 24.717082977294922, 24.7..."
2024-03-01 00:10:00,"[[25.669565200805664, 25.669565200805664, 25.6..."
2024-03-01 00:15:00,"[[24.44557762145996, 24.59754180908203, 23.801..."
2024-03-01 00:20:00,"[[22.925626754760742, 22.925626754760742, 24.1..."
...,...
2024-12-30 23:35:00,"[[26.173906326293945, 27.259140014648438, 27.2..."
2024-12-30 23:40:00,"[[27.0389347076416, 27.0389347076416, 27.03893..."
2024-12-30 23:45:00,"[[26.902502059936523, 26.72080421447754, 25.94..."


In [115]:
df_nagoya_maps_2024.to_pickle("./df_nagoya_maps_2024.pkl")

In [146]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [147]:
%cp df_nagoya_maps_2024.pkl /content/drive/MyDrive/df_nagoya_maps_2024.pkl

In [116]:
np.array(df_nagoya_maps_2022_2024.iloc[0]['TECMAP'])

array([[15.24487209, 15.35594749, 15.62724018, 15.62724018, 15.62724018,
        15.62724018, 15.62724018, 15.67634392, 14.8411026 , 14.8411026 ,
        14.8411026 , 14.8411026 , 14.8411026 , 14.98174191, 14.98174191,
        14.98174191, 14.59172058, 14.59172058, 14.67130184, 14.91833305,
        13.34468651, 13.34468651, 13.34468651, 13.6588335 , 13.6588335 ,
        12.99385834, 13.52103233, 13.52103233, 13.52103233, 13.52103233,
        13.11270905, 13.3315134 , 13.3315134 , 13.73480034, 13.73480034,
        13.73480034, 13.73480034, 13.73480034, 13.73480034, 13.73480034,
        13.73480034, 13.73480034, 13.37285614, 14.27384949, 13.73251057,
        13.73251057, 13.43309975, 13.57339764, 13.57339764, 12.4174366 ,
        12.4174366 , 11.81858444, 11.81858444, 11.81858444, 12.12085342,
        12.12085342, 11.11243725, 11.11243725, 11.11243725, 11.11243725,
        11.11243725, 11.11243725, 11.11243725, 11.11243725, 11.11243725,
        11.260355  , 11.260355  , 11.31899071, 11.3

In [117]:
np.array(df_nagoya_maps_2022_2024.iloc[0]['TECMAP']).shape

(242, 221)

In [118]:
df_nagoya_maps_2022_2024_0800 = df_nagoya_maps_2022_2024.between_time('08:00', '08:00')

df_nagoya_maps_2022_2024_1600 = df_nagoya_maps_2022_2024.between_time('16:00', '16:00')

df_nagoya_maps_2022_2024_2000_2200_0000_0200_0400 = pd.concat([
    df_nagoya_maps_2022_2024.between_time('20:00', '20:00'),
    df_nagoya_maps_2022_2024.between_time('22:00', '22:00'),
    df_nagoya_maps_2022_2024.between_time('00:00', '00:00'),
    df_nagoya_maps_2022_2024.between_time('02:00', '02:00'),
    df_nagoya_maps_2022_2024.between_time('04:00', '04:00')
])

df_nagoya_maps_2022_2024_2000_2200_0000_0200_0400.sort_index(inplace=True)

print("Data from 08:00:")
print(df_nagoya_maps_2022_2024_0800)

print("\nData from 16:00:")
print(df_nagoya_maps_2022_2024_1600)

print("\nData from 20:00 to 04:00:")
print(df_nagoya_maps_2022_2024_2000_2200_0000_0200_0400)

Data from 08:00:
                                                                TECMAP
DATETIME                                                              
2022-03-01 08:00:00  [[8.598779678344727, 8.598779678344727, 8.5987...
2022-03-02 08:00:00  [[11.66019058227539, 11.66019058227539, 11.660...
2022-03-03 08:00:00  [[11.679936408996582, 11.679936408996582, 11.6...
2022-03-04 08:00:00  [[12.4481840133667, 12.4481840133667, 12.36721...
2022-03-05 08:00:00  [[10.615665435791016, 9.959951400756836, 9.959...
...                                                                ...
2024-12-26 08:00:00  [[21.108211517333984, 21.108211517333984, 21.1...
2024-12-27 08:00:00  [[17.90998077392578, 17.90998077392578, 17.909...
2024-12-28 08:00:00  [[23.005971908569336, 23.005971908569336, 23.0...
2024-12-29 08:00:00  [[29.83434295654297, 29.83434295654297, 30.000...
2024-12-30 08:00:00  [[19.819252014160156, 19.819252014160156, 18.0...

[301 rows x 1 columns]

Data from 16:00:
                  

In [119]:
np.shape(np.array(df_nagoya_maps_2022_2024_0800.iloc[:]['TECMAP']))

(301,)

In [120]:
nagoya_maps_2022_2024_0800 = np.array(df_nagoya_maps_2022_2024_0800.iloc[:]['TECMAP'])

In [121]:
np.shape(nagoya_maps_2022_2024_0800)

(301,)

In [122]:
np_nagoya_maps_2022_2024_0800 = []
for i in range(len(nagoya_maps_2022_2024_0800)):
    np_nagoya_maps_2022_2024_0800.append(nagoya_maps_2022_2024_0800[i])
np_nagoya_maps_2022_2024_0800 = np.array(np_nagoya_maps_2022_2024_0800)

In [123]:
np.shape(np_nagoya_maps_2022_2024_0800)

(301, 242, 221)

In [124]:
type(np_nagoya_maps_2022_2024_0800)

numpy.ndarray

In [125]:
nagoya_maps_2022_2024_1600 = np.array(df_nagoya_maps_2022_2024_1600.iloc[:]['TECMAP'])

In [126]:
np.shape(nagoya_maps_2022_2024_1600)

(301,)

In [127]:
np_nagoya_maps_2022_2024_1600 = []
for i in range(len(nagoya_maps_2022_2024_1600)):
    np_nagoya_maps_2022_2024_1600.append(nagoya_maps_2022_2024_1600[i])
np_nagoya_maps_2022_2024_1600 = np.array(np_nagoya_maps_2022_2024_1600)

In [128]:
np.shape(np_nagoya_maps_2022_2024_1600)

(301, 242, 221)

In [129]:
type(np_nagoya_maps_2022_2024_1600)

numpy.ndarray

In [130]:
nagoya_maps_2022_2024_1600 = np.array(df_nagoya_maps_2022_2024_1600.iloc[:]['TECMAP'])

In [131]:
np.shape(nagoya_maps_2022_2024_1600)

(301,)

In [132]:
np_nagoya_maps_2022_2024_1600 = []
for i in range(len(nagoya_maps_2022_2024_1600)):
    np_nagoya_maps_2022_2024_1600.append(nagoya_maps_2022_2024_1600[i])
np_nagoya_maps_2022_2024_1600 = np.array(np_nagoya_maps_2022_2024_1600)

In [133]:
np.shape(np_nagoya_maps_2022_2024_1600)

(301, 242, 221)

In [134]:
type(np_nagoya_maps_2022_2024_1600)

numpy.ndarray

In [135]:
nagoya_maps_2022_2024_2000_2200_0000_0200_0400 = np.array(df_nagoya_maps_2022_2024_2000_2200_0000_0200_0400.iloc[:]['TECMAP'])

In [136]:
np.shape(nagoya_maps_2022_2024_2000_2200_0000_0200_0400)

(1505,)

In [137]:
np_nagoya_maps_2022_2024_2000_2200_0000_0200_0400 = []
for i in range(len(nagoya_maps_2022_2024_2000_2200_0000_0200_0400)):
    np_nagoya_maps_2022_2024_2000_2200_0000_0200_0400.append(nagoya_maps_2022_2024_2000_2200_0000_0200_0400[i])
np_nagoya_maps_2022_2024_2000_2200_0000_0200_0400 = np.array(np_nagoya_maps_2022_2024_2000_2200_0000_0200_0400)

In [138]:
np.shape(np_nagoya_maps_2022_2024_2000_2200_0000_0200_0400)

(1505, 242, 221)

In [139]:
type(np_nagoya_maps_2022_2024_2000_2200_0000_0200_0400)

numpy.ndarray

In [140]:
nagoya_maps_2022_2024_2000_2200_0000_0200_0400 = np.array(df_nagoya_maps_2022_2024_2000_2200_0000_0200_0400.iloc[:]['TECMAP'])

In [141]:
np.shape(nagoya_maps_2022_2024_2000_2200_0000_0200_0400)

(1505,)

In [142]:
np_nagoya_maps_2022_2024_2000_2200_0000_0200_0400 = []
for i in range(len(nagoya_maps_2022_2024_2000_2200_0000_0200_0400)):
    np_nagoya_maps_2022_2024_2000_2200_0000_0200_0400.append(nagoya_maps_2022_2024_2000_2200_0000_0200_0400[i])
np_nagoya_maps_2022_2024_2000_2200_0000_0200_0400 = np.array(np_nagoya_maps_2022_2024_2000_2200_0000_0200_0400)

In [143]:
np.shape(np_nagoya_maps_2022_2024_2000_2200_0000_0200_0400)

(1505, 242, 221)

In [144]:
type(np_nagoya_maps_2022_2024_2000_2200_0000_0200_0400)

numpy.ndarray

# TF1 adjusts

## Continuous

In [ ]:
np.array(df1.iloc[0]['TECMAP'])

In [ ]:
np.array(df1.iloc[0]['TECMAP']).shape

In [ ]:
df1_0800 = df1.between_time('08:00', '08:00')

df1_1600 = df1.between_time('16:00', '16:00')

df1_2000_2200_0000_0200_0400 = pd.concat([
    df1.between_time('20:00', '20:00'),
    df1.between_time('22:00', '22:00'),
    df1.between_time('00:00', '00:00'),
    df1.between_time('02:00', '02:00'),
    df1.between_time('04:00', '04:00')
])

df1_2000_2200_0000_0200_0400.sort_index(inplace=True)

print("Data from 08:00:")
print(df1_0800)

print("\nData from 16:00:")
print(df1_1600)

print("\nData from 20:00 to 04:00:")
print(df1_2000_2200_0000_0200_0400)

In [ ]:
np.shape(np.array(df1.iloc[:]['TECMAP']))

In [ ]:
maps1 = np.array(df1.iloc[:]['TECMAP'])

In [ ]:
np.shape(df1)

In [ ]:
np_maps1 = []
for i in range(len(maps1)):
    np_maps1.append(maps1[i])
np_maps1 = np.array(np_maps1)

In [ ]:
np.shape(np_maps1)

In [ ]:
type(np_maps1)

## 08:00 UT





In [ ]:
df1_0800.to_pickle("./TF1_Nagoya_TEC_maps_2022_2024_0800.pkl")

In [ ]:
np.shape(np.array(df1_0800.iloc[:]['TECMAP']))

In [ ]:
maps1_0800 = np.array(df1_0800.iloc[:]['TECMAP'])

In [ ]:
np.shape(df1_0800)

In [ ]:
np_maps1_0800 = []
for i in range(len(maps1_0800)):
    np_maps1_0800.append(maps1_0800[i])
np_maps1_0800 = np.array(np_maps1_0800)

In [ ]:
np.shape(np_maps1_0800)

In [ ]:
type(np_maps1_0800)

In [ ]:
np.save('TF1_Nagoya_TEC_maps_2022_2024_0800.npy', np_maps1_0800)

## 16:00 UT

In [ ]:
df1_1600.to_pickle("./TF1_Nagoya_TEC_maps_2022_2024_1600.pkl")

In [ ]:
maps1_1600 = np.array(df1_1600.iloc[:]['TECMAP'])

In [ ]:
np.shape(maps1_1600)

In [ ]:
np_maps1_1600 = []
for i in range(len(maps1_1600)):
    np_maps1_1600.append(maps1_1600[i])
np_maps1_1600 = np.array(np_maps1_1600)

In [ ]:
np.shape(np_maps1_1600)

In [ ]:
type(np_maps1_1600)

In [ ]:
np.save('TF1_Nagoya_TEC_maps_2022_2024_1600.npy', np_maps1_1600)

## 20:00 to 04:00 UT

In [ ]:
df1_2000_2200_0000_0200_0400.to_pickle("./TF1_Nagoya_TEC_maps_2022_2024_2000_2200_0000_0200_0400.pkl")

In [ ]:
maps1_2000_2200_0000_0200_0400 = np.array(df1_2000_2200_0000_0200_0400.iloc[:]['TECMAP'])

In [ ]:
np.shape(maps1_2000_2200_0000_0200_0400)

In [ ]:
np_maps1_2000_2200_0000_0200_0400 = []
for i in range(len(maps1_2000_2200_0000_0200_0400)):
    np_maps1_2000_2200_0000_0200_0400.append(maps1_2000_2200_0000_0200_0400[i])
np_maps1_2000_2200_0000_0200_0400 = np.array(np_maps1_2000_2200_0000_0200_0400)

In [ ]:
np.shape(np_maps1_2000_2200_0000_0200_0400)

In [ ]:
type(np_maps1_2000_2200_0000_0200_0400)

In [ ]:
np.save('TF1_Nagoya_TEC_maps_2022_2024_2000_2200_0000_0200_0400.npy', np_maps1_2000_2200_0000_0200_0400)

# TF2 adjusts

## Continuous

In [ ]:
np.array(df3.iloc[0]['TECMAP'])

array([[25.81098938, 25.53123283, 25.53123283, 25.10790443, 25.10790443,
        25.12040329, 24.5556488 , 24.5556488 , 24.5556488 , 23.49295998,
        24.44730377, 25.39146423, 25.39146423, 25.58919716, 24.60969543,
        24.29973602, 24.29973602, 23.48611069, 23.48611069, 24.8562851 ,
        24.8562851 , 25.569561  , 25.569561  , 24.14855003, 24.14855003,
        24.14855003, 24.14855003, 24.14855003, 24.14855003, 24.14855003,
        24.14855003, 23.93475151, 23.93475151, 23.054039  , 21.54585648,
        21.54585648, 22.31619072, 22.31619072, 22.90351868, 22.90351868,
        23.57149506, 23.57149506, 23.57149506, 22.20942307, 20.65611076,
        20.65611076, 21.86031532, 21.86031532, 21.86031532, 21.86031532,
        21.86031532, 22.22672462, 22.22672462, 22.22672462, 22.0849781 ,
        22.0849781 , 22.0849781 , 22.0849781 , 22.0849781 , 21.27880669,
        22.5953598 , 22.5953598 , 22.36440277, 23.78794861, 23.78794861,
        23.78794861, 25.19330978, 25.19330978, 25.1

In [ ]:
np.array(df3.iloc[0]['TECMAP']).shape

In [ ]:
df3_0800 = df3.between_time('08:00', '08:00')

df3_1600 = df3.between_time('16:00', '16:00')

df3_2000_0400 = pd.concat([
    df3.between_time('20:00', '20:00'),
    df3.between_time('20:30', '20:30'),
    df3.between_time('21:00', '21:00'),
    df3.between_time('21:30', '21:30'),
    df3.between_time('22:00', '22:00'),
    df3.between_time('22:30', '22:30'),
    df3.between_time('23:00', '23:00'),
    df3.between_time('23:30', '23:30'),
    df3.between_time('00:00', '00:00'),
    df3.between_time('00:30', '00:30'),
    df3.between_time('01:00', '01:00'),
    df3.between_time('01:30', '01:30'),
    df3.between_time('02:00', '02:00'),
    df3.between_time('02:30', '02:30'),
    df3.between_time('03:00', '03:00'),
    df3.between_time('03:30', '03:30'),
    df3.between_time('04:00', '04:00')
])

df3_2000_0400.sort_index(inplace=True)

print("Data from 08:00:")
print(df3_0800)

print("\nData from 16:00:")
print(df3_1600)

print("\nData from 20:00 to 04:00:")
print(df3_2000_0400)

In [ ]:
np.shape(np.array(df3.iloc[:]['TECMAP']))

In [ ]:
maps3 = np.array(df3.iloc[:]['TECMAP'])

In [ ]:
np.shape(df3)

In [ ]:
np_maps3 = []
for i in range(len(maps3)):
    np_maps3.append(maps3[i])
np_maps3 = np.array(np_maps3)

In [ ]:
np.shape(np_maps3)

In [ ]:
type(np_maps3)

## 08:00 UT

In [ ]:
df3_0800.to_pickle("./TF3_Nagoya_TEC_maps_2024_0800.pkl")

In [ ]:
np.shape(np.array(df3_0800.iloc[:]['TECMAP']))

In [ ]:
maps3_0800 = np.array(df3_0800.iloc[:]['TECMAP'])

In [ ]:
np.shape(df3_0800)

In [ ]:
np_maps3_0800 = []
for i in range(len(maps3_0800)):
    np_maps3_0800.append(maps3_0800[i])
np_maps3_0800 = np.array(np_maps3_0800)

In [ ]:
np.shape(np_maps3_0800)

In [ ]:
type(np_maps3_0800)

In [ ]:
np.save('TF3_Nagoya_TEC_maps_2024_0800.npy', np_maps3_0800)

## 16:00 UT

In [ ]:
df3_1600.to_pickle("./TF3_Nagoya_TEC_maps_2024_1600.pkl")

In [ ]:
maps3_1600 = np.array(df3_1600.iloc[:]['TECMAP'])

In [ ]:
np.shape(maps3_1600)

In [ ]:
np_maps3_1600 = []
for i in range(len(maps3_1600)):
    np_maps3_1600.append(maps3_1600[i])
np_maps3_1600 = np.array(np_maps3_1600)

In [ ]:
np.shape(np_maps3_1600)

In [ ]:
type(np_maps3_1600)

In [ ]:
np.save('TF3_Nagoya_TEC_maps_2024_1600.npy', np_maps3_1600)

## 20:00 to 04:00 UT

In [ ]:
df3_2000_0400.to_pickle("./TF3_Nagoya_TEC_maps_2024_2000_0400.pkl")

In [ ]:
df3_2000_0400

In [ ]:
maps3_2000_0400 = np.array(df3_2000_0400.iloc[:]['TECMAP'])

In [ ]:
np.shape(maps3_2000_0400)

In [ ]:
np_maps3_2000_0400 = []
for i in range(len(maps3_2000_0400)):
    np_maps3_2000_0400.append(maps3_2000_0400[i])
np_maps3_2000_0400 = np.array(np_maps3_2000_0400)

In [ ]:
np.shape(np_maps3_2000_0400)

In [ ]:
type(np_maps3_2000_0400)

In [ ]:
np.save('TF3_Nagoya_TEC_maps_2024_2000_0400.npy', np_maps3_2000_0400)

# Download outputs

In [ ]:
from google.colab import files

In [ ]:
%pwd

In [ ]:
%cd '/content/Nagoya_TEC_maps_intersection_raw_files_2022_2024'

In [ ]:
!ls -lh Nagoya*

In [ ]:
files.download('Nagoya_TEC_maps_intersection_2024.npy')

In [ ]:
%pwd

In [ ]:
%cd '/content/Nagoya_TEC_maps_intersection_raw_files_2022_2024'

In [ ]:
%pwd

In [ ]:
!ls -lh TF*.npy

In [ ]:
files.download('./TF1_Nagoya_TEC_maps_2022_2024_0800.pkl')

In [ ]:
files.download('./TF1_Nagoya_TEC_maps_2022_2024_1600.pkl')

In [ ]:
files.download('./TF1_Nagoya_TEC_maps_2022_2024_2000_2200_0000_0200_0400.pkl')

In [ ]:
files.download('./TF2_Nagoya_TEC_maps_2022_2024_0800.pkl')

In [ ]:
files.download('./TF2_Nagoya_TEC_maps_2022_2024_1600.pkl')

In [ ]:
files.download('./TF2_Nagoya_TEC_maps_2022_2024_2000_2200_0000_0200_0400.pkl')

In [ ]:
!ls -lh TF3*

In [ ]:
%cp TF3_Nagoya_TEC_maps_2024_2000_0400.pkl /content/drive/MyDrive/TF3_Nagoya_TEC_maps_2024_2000_0400.pkl

In [ ]:
files.download('./TF1_Nagoya_TEC_maps_2022_2024_0800.npy')

In [ ]:
files.download('./TF1_Nagoya_TEC_maps_2022_2024_1600.npy')

In [ ]:
files.download('./TF1_Nagoya_TEC_maps_2022_2024_2000_2200_0000_0200_0400.npy')

In [ ]:
files.download('./TF2_Nagoya_TEC_maps_2022_2024_0800.npy')

In [ ]:
files.download('./TF2_Nagoya_TEC_maps_2022_2024_1600.npy')

In [ ]:
files.download('./TF2_Nagoya_TEC_maps_2022_2024_2000_2200_0000_0200_0400.npy')

In [ ]:
!ls -lh TF3*

In [ ]:
%cp TF3_Nagoya_TEC_maps_2024_0800.npy /content/drive/MyDrive/TF3_Nagoya_TEC_maps_2024_0800.npy

In [ ]:
%cp TF3_Nagoya_TEC_maps_2024_1600.npy /content/drive/MyDrive/TF3_Nagoya_TEC_maps_2024_1600.npy

In [ ]:
%cp TF3_Nagoya_TEC_maps_2024_2000_0400.npy /content/drive/MyDrive/TF3_Nagoya_TEC_maps_2024_2000_0400.npy